# 04 · Grad-CAM++: where does each model look?

Phase 2. For **every trained model**, on the same test images from FF++ and the unseen Celeb-DF:

1. **Facial regions per face:** eyes, nose, mouth, cheeks, forehead and chin are placed from the face detector's
   5 landmarks (eyes, nose tip, mouth corners), not a fixed grid, so region claims are accurate on every face.
2. **Heatmap gallery:** Grad-CAM++ for every model on the same images.
3. **Are the heatmaps faithful?** Deletion / insertion test (Petsiuk et al., RISE, 2018): perturb pixels in
   the order the heatmap ranks them and watch the model's confidence. Grad-CAM vs. Grad-CAM++ vs. a random map,
   with paired Wilcoxon tests.
4. **Region attribution:** which facial regions each model relies on, per manipulation method and on Celeb-DF.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 · **Internet:** On
- **Inputs:** **01_extract_faces** (FF++), **01b_extract_celebdf**, and **02_train** (plus any copies of 02)

**Produces:** `/kaggle/working/outputs/gradcam/` (CSV tables + figures).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup, find_datasets, checkpoint_roots
setup(data=False)

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from src.classifier.compare import find_checkpoints
from src.classifier.model import BASELINES
from src.config import IMG_SIZE
from src.faces import FaceCropper
from src.xai.gradcam import Explainer, FACE_REGIONS, draw_bbox, draw_regions, landmark_regions

DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
ROOTS = checkpoint_roots()
CKPTS = find_checkpoints(ROOTS)
print("datasets:", DATASETS)
print("models:", list(CKPTS))
assert "FF++" in DATASETS and CKPTS, "Add 01_extract_faces and 02_train as inputs"
OUT = "/kaggle/working/outputs/gradcam"
os.makedirs(OUT, exist_ok=True)

## Settings
`N` = images per dataset for the faithfulness and region analysis (one frame per test video, balanced across manipulation methods). Every model sees the same images.

In [ ]:
N = 300
CAM_METHODS = ["gradcam++", "gradcam", "random"]

## 1 · Facial-region placement (sanity check)
Region boxes from the 5 landmarks on random test faces. Check that eyes, nose and mouth boxes sit on the right features.

In [ ]:
faces = FaceCropper()
picks = []
for ds, root in DATASETS.items():
    files = sorted(Path(root, "test").glob("*/*.jpg"))
    rng = np.random.default_rng(0)
    picks += [(ds, files[i]) for i in rng.choice(len(files), 5, replace=False)]
fig, axes = plt.subplots(1, len(picks), figsize=(2.4 * len(picks), 2.8))
for ax, (ds, p) in zip(axes, picks):
    rgb = np.asarray(Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
    found = faces.landmarks(rgb)
    regions = landmark_regions(*found, IMG_SIZE, IMG_SIZE) if found is not None else FACE_REGIONS
    ax.imshow(draw_regions(rgb, regions)); ax.axis("off")
    ax.set_title(f"{ds}\n{'landmarks' if found is not None else 'grid (no face)'}", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/region_placement.png", dpi=130); plt.show()
print("colours: left eye red, right eye orange, nose yellow, mouth green, cheeks blue, forehead purple, chin white")

## 2 · Heatmap gallery
The same images for every model: one fake per FF++ manipulation, one FF++ real, and one real + one fake from Celeb-DF. Title = the model's verdict, P(fake) and the region with the most activation.

In [ ]:
gallery = []
for ds, root in DATASETS.items():
    files = sorted(Path(root, "test").glob("*/*.jpg"))
    by_method = {}
    for p in files:
        by_method.setdefault(p.name.split("__")[0], []).append(p)
    rng = np.random.default_rng(1)
    gallery += [(ds, m, ps[rng.integers(len(ps))]) for m, ps in sorted(by_method.items())]

fig, axes = plt.subplots(len(CKPTS) + 1, len(gallery), figsize=(2.3 * len(gallery), 2.5 * (len(CKPTS) + 1)))
for j, (ds, m, p) in enumerate(gallery):
    axes[0, j].imshow(Image.open(p).resize((IMG_SIZE, IMG_SIZE)))
    axes[0, j].set_title(f"{ds}\n{'real' if m == 'original' else m}", fontsize=8)
for i, (backbone, path) in enumerate(CKPTS.items(), start=1):
    explain = Explainer(path)
    for j, (ds, m, p) in enumerate(gallery):
        e = explain(Image.open(p))
        axes[i, j].imshow(draw_bbox(e.overlay, e.bbox))
        axes[i, j].set_title(f"{e.label} {e.prob_fake:.2f} · {e.top_regions[0][0]}", fontsize=7)
    axes[i, 0].set_ylabel(BASELINES.get(backbone, backbone), fontsize=8)
    del explain
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT}/gallery.png", dpi=130); plt.show()

## 3 · Faithfulness and region attribution
Runs every model × every CAM method on the same images. Roughly 2–5 minutes per model.

In [ ]:
roots = " ".join(f'"{r}"' for r in ROOTS)
datasets = " ".join(f'"{k}={v}"' for k, v in DATASETS.items())
!TQDM_MININTERVAL=60 python -m src.xai.evaluate_cam --roots {roots} --datasets {datasets} --n {N} --methods {" ".join(CAM_METHODS)}

### Deletion / insertion
**Deletion AUC: lower is better** (removing the highlighted pixels quickly kills the model's confidence). **Insertion AUC: higher is better** (restoring them quickly brings it back). A method is only useful if it beats `random`.

In [ ]:
summary = pd.read_csv(f"{OUT}/faithfulness_summary.csv")
for metric in ("deletion_auc", "insertion_auc"):
    display(Markdown(f"**{metric}**"))
    display(summary.pivot_table(index=["dataset", "model"], columns="cam", values=metric).round(4))

curves = pd.read_csv(f"{OUT}/curves.csv")
for ds in curves.dataset.unique():
    models = sorted(curves.model.unique())
    fig, axes = plt.subplots(2, len(models), figsize=(2.6 * len(models), 5), sharey=True)
    for j, mdl in enumerate(models):
        for cam, g in curves[(curves.dataset == ds) & (curves.model == mdl)].groupby("cam"):
            axes[0, j].plot(g.fraction, g.deletion, label=cam)
            axes[1, j].plot(g.fraction, g.insertion, label=cam)
        axes[0, j].set_title(mdl, fontsize=8)
    axes[0, 0].set_ylabel("deletion: P(pred)"); axes[1, 0].set_ylabel("insertion: P(pred)")
    axes[0, 0].legend(fontsize=7)
    for ax in axes[1]: ax.set_xlabel("fraction of pixels")
    plt.suptitle(f"{ds}: deletion (top) and insertion (bottom) curves"); plt.tight_layout()
    plt.savefig(f"{OUT}/curves_{ds}.png", dpi=130); plt.show()

### Is Grad-CAM++ better than Grad-CAM, and both better than random?
Paired Wilcoxon signed-rank test per model and dataset (same images). p < 0.05 = significant.

In [ ]:
tests = pd.read_csv(f"{OUT}/faithfulness_tests.csv")
tests["significant"] = tests.p_value < 0.05
for metric in ("deletion_auc", "insertion_auc"):
    display(Markdown(f"**{metric}**"))
    display(tests[tests.metric == metric].drop(columns="metric").round(4))

### Region attribution
Mean share of Grad-CAM++ activation per facial region, over correctly classified test images. Rows = models; one panel per manipulation.

In [ ]:
attr = pd.read_csv(f"{OUT}/region_attribution.csv")
regions = [c for c in attr.columns if c not in ("model", "dataset", "manipulation")]
panels = attr[["dataset", "manipulation"]].drop_duplicates().values.tolist()
fig, axes = plt.subplots(len(panels), 1, figsize=(11, 2.6 * len(panels)))
for ax, (ds, manip) in zip(np.atleast_1d(axes), panels):
    t = attr[(attr.dataset == ds) & (attr.manipulation == manip)].set_index("model")[regions]
    im = ax.imshow(t.values, cmap="magma", vmin=0, vmax=max(0.4, t.values.max()))
    ax.set_yticks(range(len(t)), t.index, fontsize=8)
    ax.set_xticks(range(len(regions)), regions, rotation=25, ha="right", fontsize=8)
    for (r, c), v in np.ndenumerate(t.values):
        ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v < 0.25 else "black")
    ax.set_title(f"{ds} · {'real' if manip == 'original' else manip}", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/region_attribution.png", dpi=130); plt.show()
display(attr.round(3))

In [ ]:
regions_df = pd.read_csv(f"{OUT}/regions.csv")
share = (regions_df.region_source == "landmarks").mean()
print(f"facial regions from landmarks on {share:.0%} of images (rest: fixed grid)")